# 03 — DuckDB Data Exploration

**From Clinical Case Reports to Knowledge Graphs**

Explores `data/duckdb/clinical_cases.duckdb`, built by
[`01_data_preparation.ipynb`](01_data_preparation.ipynb), directly as a
native DuckDB database (`cases`, `metadata`, `data_dictionary` tables) rather
than through CSV. The same SQL used here also drives
[`02a_csv_data_exploraton_sample.ipynb`](02a_csv_data_exploraton_sample.ipynb)
and
[`02b_csv_data_exploraton_full.ipynb`](02b_csv_data_exploraton_full.ipynb)
against the CSV exports — compare against those to see the effect of
sampling and of DuckDB's native typed storage versus re-parsed CSV text
(e.g. `authors`/`mesh_terms` stay real `VARCHAR[]` arrays here, and `year`
stays `VARCHAR`, per `docs/data_source.md`).


## 1. Setup


In [1]:
from pathlib import Path

import duckdb

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent
DB_PATH = NOTEBOOK_DIR.parent / "data" / "duckdb" / "clinical_cases.duckdb"

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"{DB_PATH} not found — run 01_data_preparation.ipynb first."
    )

con = duckdb.connect(str(DB_PATH), read_only=True)
con.sql("SHOW TABLES")


┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ cases           │
│ data_dictionary │
│ metadata        │
└─────────────────┘

## 2. Example queries — exploring the tables


## Table schema


In [2]:
con.sql("DESCRIBE cases")


┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ article_id  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ age         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ case_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ case_text   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ gender      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [3]:
con.sql("DESCRIBE metadata")


┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ article_id       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ authors          │ VARCHAR[]   │ YES     │ NULL    │ NULL    │ NULL    │
│ case_amount      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ doi              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ journal          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ journal_detail   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keywords         │ VARCHAR[]   │ YES     │ NULL    │ NULL    │ NULL    │
│ license          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ link             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ major_mesh_terms │ VARC

### A random sample of cases


In [4]:
con.sql("SELECT * FROM cases LIMIT 5")


┌────────────┬────────┬───────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [5]:
con.sql("SELECT * FROM metadata LIMIT 5")


┌────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┬─────────────────────────────┬─────────────────────────────────────┬────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────┬───────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

### Text length distribution


In [6]:
con.sql("""
    SELECT
        MIN(LENGTH(case_text)) AS min_chars,
        MEDIAN(LENGTH(case_text)) AS median_chars,
        AVG(LENGTH(case_text))::INT AS avg_chars,
        MAX(LENGTH(case_text)) AS max_chars
    FROM cases
""")


┌───────────┬──────────────┬───────────┬───────────┐
│ min_chars │ median_chars │ avg_chars │ max_chars │
│   int64   │    double    │   int32   │   int64   │
├───────────┼──────────────┼───────────┼───────────┤
│         9 │       2543.0 │      2965 │     79243 │
└───────────┴──────────────┴───────────┴───────────┘

### Patient demographics


In [7]:
con.sql("""
    SELECT gender, COUNT(*) AS n_cases
    FROM cases
    GROUP BY gender
    ORDER BY n_cases DESC
""")


┌─────────────┬─────────┐
│   gender    │ n_cases │
│   varchar   │  int64  │
├─────────────┼─────────┤
│ Male        │   47069 │
│ Female      │   44778 │
│ Unknown     │    6679 │
│ Transgender │     115 │
└─────────────┴─────────┘

In [8]:
con.sql("""
    SELECT
        MIN(age) AS min_age,
        MEDIAN(age) AS median_age,
        AVG(age)::INT AS avg_age,
        MAX(age) AS max_age
    FROM cases
    WHERE age IS NOT NULL
""")


┌─────────┬────────────┬─────────┬─────────┐
│ min_age │ median_age │ avg_age │ max_age │
│ double  │   double   │  int32  │ double  │
├─────────┼────────────┼─────────┼─────────┤
│     0.0 │       42.0 │      41 │   120.0 │
└─────────┴────────────┴─────────┴─────────┘

### Joining `cases` with `metadata`

`cases` and `metadata` share the `article_id` (PMCID) column, so we can
bring in publication year, journal, license, etc.


In [9]:
con.sql("""
    SELECT m.year, COUNT(*) AS n_cases
    FROM cases AS c
    JOIN metadata AS m USING (article_id)
    GROUP BY m.year
    ORDER BY m.year
""")


┌─────────┬─────────┐
│  year   │ n_cases │
│ varchar │  int64  │
├─────────┼─────────┤
│ 1990    │      11 │
│ 1991    │       7 │
│ 1992    │       5 │
│ 1993    │      10 │
│ 1994    │       9 │
│ 1995    │       9 │
│ 1996    │      10 │
│ 1997    │      16 │
│ 1998    │      13 │
│ 1999    │      14 │
│  ·      │       · │
│  ·      │       · │
│  ·      │       · │
│ 2017    │    5905 │
│ 2018    │    6734 │
│ 2019    │    7359 │
│ 2020    │    7562 │
│ 2021    │    8120 │
│ 2022    │    8715 │
│ 2023    │   11616 │
│ 2024    │    8893 │
│ 2025    │    6513 │
│ 2026    │    2250 │
└─────────┴─────────┘
       37 rows     
     (20 shown)     

In [10]:
con.sql("""
    SELECT journal, COUNT(DISTINCT article_id) AS n_articles
    FROM metadata
    GROUP BY journal
    ORDER BY n_articles DESC
    LIMIT 10
""")


┌────────────────────────┬────────────┐
│        journal         │ n_articles │
│        varchar         │   int64    │
├────────────────────────┼────────────┤
│ Front Oncol            │       2434 │
│ Cureus                 │       2413 │
│ Surg Neurol Int        │       2359 │
│ SAGE Open Med Case Rep │       1983 │
│ Pan Afr Med J          │       1820 │
│ Case Rep Med           │       1726 │
│ Front Pediatr          │       1434 │
│ Front Med (Lausanne)   │       1404 │
│ Int J Surg Case Rep    │       1375 │
│ Front Neurol           │       1269 │
└────────────────────────┴────────────┘
  10 rows                   2 columns

In [11]:
con.sql("""
    SELECT license, COUNT(*) AS n_articles
    FROM metadata
    GROUP BY license
    ORDER BY n_articles DESC
""")


┌─────────────┬────────────┐
│   license   │ n_articles │
│   varchar   │   int64    │
├─────────────┼────────────┤
│ CC BY       │      49838 │
│ CC BY-NC    │      16648 │
│ CC BY-NC-SA │       9598 │
│ CC0         │         53 │
└─────────────┴────────────┘

## Wrap up


In [12]:
con.close()
